# Film Factory — GPU Notebook

Three compute-heavy tasks moved off local CPU to Colab Pro A100:

| Section | What it does | Typical cost (A100 80GB) |
|---------|-------------|---------------------|
| **VOICE** | XTTS-v2 voice cloning from your reference audio | ~0.05 units / episode |
| **SAM2**  | Segment images into bg/mid/subject layers | ~0.10 units / 20 images |
| **MOTION**| SVD-XT: animate images into 4s parallax clips | ~4 units / 17 images |

**Budget reference:** A100 ≈ 7 units/hour · ~100 units/month (~14 hrs total)  
Voice + SAM2 together: ~0.15 units per episode.  
SVD-XT: ~4 units — only run when images are final.

### One-time setup (do this once)
1. Upload your reference audio file to Drive at the path set in `REFERENCE_AUDIO_PATH` below
2. Run cell 1 (Drive mount) + cell 2 (config) + cell 3 (deps) once  
   → XTTS-v2 model downloads to Drive (~1.8 GB, ~2 min)  
   → Every future session loads from Drive (~10 seconds)

### Per-run workflow
1. Run local pipeline through Agent 4a → produces `outputs/colab_input/episode_N_chunks.json`
2. Sync `outputs/` to your Drive folder
3. Toggle `RUN_VOICE` / `RUN_SAM2` / `RUN_MOTION` in cell 2
4. Run notebook top to bottom
5. Sync Drive `outputs/` back to local machine
6. Set `VOICE_SOURCE=colab` and/or `SAM2_SOURCE=colab` in local `.env`
7. Re-run local pipeline from Agent 4 onward

In [ ]:
# @title 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# @title 2. Configuration and ON/OFF switches
import os

# ── DRIVE ROOT ────────────────────────────────────────────────────────────────
DRIVE_OUTPUTS = '/content/drive/MyDrive/film_factory/film_factory/outputs'

# ── ON/OFF SWITCHES ───────────────────────────────────────────────────────────
RUN_VOICE  = True    # XTTS-v2 voice cloning  — regenerate with new reference audio
RUN_SAM2   = False   # SAM2 layer segmentation — images already done, skip
RUN_MOTION = False   # SVD-XT motion clips     — skip

# ── VOICE CLONING SETTINGS ────────────────────────────────────────────────────
REFERENCE_AUDIO_PATH = '/content/drive/MyDrive/film_factory/film_factory/ElevenLabs_Text_to_Speech_audio.mp3'
MODEL_CACHE_DIR = '/content/drive/MyDrive/film_factory/film_factory/models/xtts_v2'

CHUNK_GAP_S = 0.8
TAIL_SILENCE_S = 2.0

# ── SAM2 SETTINGS ─────────────────────────────────────────────────────────────
SAM2_CHECKPOINT = 'sam2.1_hiera_small'

# ── MOTION SETTINGS ───────────────────────────────────────────────────────────
MOTION_METHOD = 'svd'
CLIP_DURATION = 4
OUTPUT_FPS    = 24
OUTPUT_W      = 1920
OUTPUT_H      = 1080

# ── DERIVED PATHS ─────────────────────────────────────────────────────────────
COLAB_INPUT_DIR   = f'{DRIVE_OUTPUTS}/colab_input'
AUDIO_OUTPUT_DIR  = f'{DRIVE_OUTPUTS}/audio'
SCENE_IMAGES_DIR  = f'{DRIVE_OUTPUTS}/scene_images'
REVEAL_IMAGES_DIR = f'{DRIVE_OUTPUTS}/reveal_images'
LAYERS_DIR        = f'{DRIVE_OUTPUTS}/layers'
MOTION_CLIPS_DIR  = f'{DRIVE_OUTPUTS}/motion_clips'

for d in [AUDIO_OUTPUT_DIR, LAYERS_DIR, MOTION_CLIPS_DIR, MODEL_CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

print('Configuration:')
print(f'  DRIVE_OUTPUTS       = {DRIVE_OUTPUTS}')
print(f'  REFERENCE_AUDIO     = {REFERENCE_AUDIO_PATH}')
print(f'  MODEL_CACHE_DIR     = {MODEL_CACHE_DIR}')
print(f'  RUN_VOICE           = {RUN_VOICE}')
print(f'  RUN_SAM2            = {RUN_SAM2}')
print(f'  RUN_MOTION          = {RUN_MOTION}')

import glob
chunks_files = sorted(glob.glob(f'{COLAB_INPUT_DIR}/episode_*_chunks.json'))
ref_exists = os.path.exists(REFERENCE_AUDIO_PATH)

print(f'\nChunk files found : {len(chunks_files)}')
for f in chunks_files:
    print(f'  {os.path.basename(f)}')
print(f'Reference audio   : {"OK found" if ref_exists else "NOT FOUND -- see instructions below"}')
if not ref_exists:
    print(f'\n  Upload your reference audio to Google Drive at this exact path:')
    print(f'    {REFERENCE_AUDIO_PATH}')
    print(f'  File name must be: ElevenLabs_Text_to_Speech_audio.mp3')
    print(f'  In Drive: My Drive -> film_factory -> film_factory -> (upload here)')

In [ ]:
# @title 3. Install dependencies
# NOTE: This cell auto-restarts the runtime once to lock numpy<2.0 into memory.
# After the restart, re-run from cell 2 — all installs are cached so it's fast.
import subprocess, sys, importlib.util

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# ── numpy<2.0 first ───────────────────────────────────────────────────────────
# Colab ships with numpy 2.x. coqui-tts requires 1.x (numpy.char removed in 2.x).
pip('numpy<2.0', 'Pillow')

# ── Voice deps ────────────────────────────────────────────────────────────────
if RUN_VOICE and importlib.util.find_spec('TTS') is None:
    # coqui-tts = Python 3.12-compatible community fork of the original TTS package.
    # Same API (from TTS.api import TTS), same XTTS-v2 model.
    # Original TTS==0.22.0 has a hard Python<3.12 constraint — won't install here.
    # PyTorch must be installed before coqui-tts (not bundled since 0.27.4).
    print('Installing coqui-tts...')
    pip('torch>=2.5.1', 'torchaudio>=2.1', 'torchvision>=0.20.1')
    pip('coqui-tts')
    pip('numpy<2.0')  # re-pin: coqui-tts dep resolution may upgrade numpy again
    print('coqui-tts installed.')
elif RUN_VOICE:
    print('coqui-tts already installed — skipping.')

# ── SAM2 deps ─────────────────────────────────────────────────────────────────
if RUN_SAM2 and importlib.util.find_spec('sam2') is None:
    # Install from PyPI pre-built wheel — no CUDA compilation step, no build errors.
    # git+https install builds from source and often fails on CUDA extension compile.
    print('Installing sam2...')
    pip('sam2')
    print('sam2 installed.')
elif RUN_SAM2:
    print('sam2 already installed — skipping.')

# ── Motion deps ───────────────────────────────────────────────────────────────
if RUN_MOTION and importlib.util.find_spec('diffusers') is None:
    pip('diffusers>=0.27', 'transformers>=4.38', 'accelerate>=0.28',
        'safetensors', 'imageio[ffmpeg]')
    if MOTION_METHOD == 'animatediff':
        pip('peft')

# ── Runtime restart to flush numpy 2.x from memory ───────────────────────────
# pip downgraded numpy on disk above, but the live session still has 2.x loaded.
# A kernel restart makes Python import the newly-installed 1.x from scratch.
# On the second run of this cell (post-restart) numpy is already 1.x — no loop.
import numpy as np
if int(np.__version__.split('.')[0]) >= 2:
    print(f'\nnumpy {np.__version__} in memory — restarting runtime to load 1.x...')
    print('Re-run from cell 2 when the runtime comes back up (installs are cached).')
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
else:
    print(f'\nnumpy {np.__version__} — all dependencies ready.')

---
## Section A — Voice Cloning (XTTS-v2)

Clones `REFERENCE_AUDIO_PATH` and synthesises every narration chunk.  
Stitches chunks into `outputs/audio/episode_N_voiceover.wav`.

**First run:** XTTS-v2 model (~1.8 GB) downloads to `MODEL_CACHE_DIR` on Drive.  
**Subsequent runs:** loads from Drive in ~10 seconds — no re-download.

In [ ]:
# @title A. Run voice cloning
if not RUN_VOICE:
    print('RUN_VOICE=False — skipping.')
else:
    import json, wave, io, struct
    import numpy as np
    import torch
    from pathlib import Path

    # Validate reference audio before loading model
    if not Path(REFERENCE_AUDIO_PATH).exists():
        ref_name = Path(REFERENCE_AUDIO_PATH).name
        ref_dir  = str(Path(REFERENCE_AUDIO_PATH).parent)
        raise FileNotFoundError(
            f'\n{"="*60}\n'
            f'Reference audio not found:\n  {REFERENCE_AUDIO_PATH}\n\n'
            f'To fix: upload "{ref_name}" to Google Drive at:\n'
            f'  My Drive → film_factory → film_factory → {ref_name}\n\n'
            f'Then re-run this cell.\n'
            f'{"="*60}'
        )

    # Point XTTS-v2 model cache to Drive so it persists across sessions
    os.environ['COQUI_TTS_HOME'] = MODEL_CACHE_DIR
    # Required: prevents XTTS-v2 from hanging on interactive license prompt
    os.environ['COQUI_TOS_AGREED'] = '1'

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Loading XTTS-v2 on {device} (model cached at {MODEL_CACHE_DIR})...')

    from TTS.api import TTS
    tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
    print('XTTS-v2 loaded.')

    SAMPLE_RATE = getattr(tts.synthesizer, 'output_sample_rate', 24000)

    def clone_chunk(text: str, reference_wav: str) -> np.ndarray:
        """
        Synthesise text using the cloned voice.
        Returns float32 numpy array at XTTS native sample rate.
        """
        wav = tts.tts(
            text=text,
            speaker_wav=reference_wav,
            language='en',
        )
        audio = np.array(wav, dtype=np.float32)
        return audio

    def float32_to_wav_bytes(audio: np.ndarray, sample_rate: int) -> bytes:
        buf = io.BytesIO()
        audio_int16 = (np.clip(audio, -1.0, 1.0) * 32767).astype(np.int16)
        with wave.open(buf, 'wb') as wf:
            wf.setnchannels(1)
            wf.setsampwidth(2)
            wf.setframerate(sample_rate)
            wf.writeframes(audio_int16.tobytes())
        return buf.getvalue()

    def stitch_wav_chunks(
        chunk_wavs: list,
        gap_s: float = CHUNK_GAP_S,
        tail_s: float = TAIL_SILENCE_S,
        sample_rate: int = SAMPLE_RATE,
    ) -> bytes:
        """Concatenate chunk WAVs with silence gaps and 2s tail silence."""
        silence_gap  = (np.zeros(int(sample_rate * gap_s),  dtype=np.int16)).tobytes()
        silence_tail = (np.zeros(int(sample_rate * tail_s), dtype=np.int16)).tobytes()
        buf = io.BytesIO()
        with wave.open(buf, 'wb') as wf:
            wf.setnchannels(1)
            wf.setsampwidth(2)
            wf.setframerate(sample_rate)
            for i, wav_bytes in enumerate(chunk_wavs):
                with wave.open(io.BytesIO(wav_bytes), 'rb') as r:
                    wf.writeframes(r.readframes(r.getnframes()))
                if i < len(chunk_wavs) - 1:
                    wf.writeframes(silence_gap)
            wf.writeframes(silence_tail)
        return buf.getvalue()

    if not chunks_files:
        print('No chunk files found. Run local pipeline to Agent 4a first.')
    else:
        for chunks_file in chunks_files:
            with open(chunks_file) as f:
                data = json.load(f)
            ep_num  = data['episode']
            chunks  = data['chunks']
            out_wav = f'{AUDIO_OUTPUT_DIR}/episode_{ep_num}_voiceover.wav'

            if Path(out_wav).exists():
                print(f'Episode {ep_num}: WAV already exists — skipping (delete to regenerate)')
                continue

            print(f'\nEpisode {ep_num}: cloning voice for {len(chunks)} chunks...')
            chunk_wavs = []

            for idx, chunk in enumerate(chunks):
                text = chunk.get('text', '').strip()
                if not text:
                    continue
                print(f'  [{idx+1}/{len(chunks)}] {len(text)} chars...')
                try:
                    audio = clone_chunk(text, REFERENCE_AUDIO_PATH)
                    chunk_wavs.append(float32_to_wav_bytes(audio, SAMPLE_RATE))
                except Exception as e:
                    print(f'    ✗ chunk {idx+1} failed: {e}')

            if not chunk_wavs:
                print(f'  ✗ No audio generated for episode {ep_num}')
                continue

            stitched = stitch_wav_chunks(chunk_wavs)
            with open(out_wav, 'wb') as f:
                f.write(stitched)
            size_kb = len(stitched) // 1024
            print(f'  ✓ Saved: {out_wav} ({size_kb} KB)')

    print('\nVoice cloning done.')

---
## Section B — SAM2 Layer Segmentation
Segments every scene and reveal image into background / midground / subject layers.  
Saves grayscale mask PNGs to `outputs/layers/` — compositor uses these for parallax depth.

GPU gives ~4× better mask quality vs CPU (more SAM2 points per side).

In [ ]:
# @title B. Run SAM2 segmentation
if not RUN_SAM2:
    print('RUN_SAM2=False — skipping.')
else:
    import torch
    import numpy as np
    import urllib.request
    from PIL import Image, ImageFilter
    from pathlib import Path
    import glob

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'SAM2 running on {device}')

    CHECKPOINT_URLS = {
        'sam2.1_hiera_tiny':      'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt',
        'sam2.1_hiera_small':     'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt',
        'sam2.1_hiera_base_plus': 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt',
        'sam2.1_hiera_large':     'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt',
    }
    # Config names relative to sam2_configs package root (PyPI install registers
    # configs as 'sam2_configs', not 'sam2.configs' or 'sam2/configs')
    CHECKPOINT_CONFIGS = {
        'sam2.1_hiera_tiny':      'sam2.1/sam2.1_hiera_t.yaml',
        'sam2.1_hiera_small':     'sam2.1/sam2.1_hiera_s.yaml',
        'sam2.1_hiera_base_plus': 'sam2.1/sam2.1_hiera_b+.yaml',
        'sam2.1_hiera_large':     'sam2.1/sam2.1_hiera_l.yaml',
    }

    # Cache SAM2 checkpoint on Drive
    sam2_cache_dir = f'{MODEL_CACHE_DIR}/../sam2'
    os.makedirs(sam2_cache_dir, exist_ok=True)
    ckpt_path = f'{sam2_cache_dir}/{SAM2_CHECKPOINT}.pt'

    if not Path(ckpt_path).exists():
        url = CHECKPOINT_URLS[SAM2_CHECKPOINT]
        print(f'Downloading SAM2 checkpoint ({SAM2_CHECKPOINT}) to Drive...')
        urllib.request.urlretrieve(url, ckpt_path)
        print(f'Downloaded: {ckpt_path}')
    else:
        print(f'SAM2 checkpoint cached: {ckpt_path}')

    config_name = CHECKPOINT_CONFIGS[SAM2_CHECKPOINT]

    from sam2.build_sam import build_sam2
    from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
    from hydra.core.global_hydra import GlobalHydra
    from hydra import initialize_config_module

    # PyPI-installed sam2 registers configs as 'sam2_configs' package (not 'sam2').
    # Clear any stale Hydra state (safe on re-run) then re-initialize correctly.
    GlobalHydra.instance().clear()
    initialize_config_module('sam2_configs', version_base='1.2')

    print(f'Loading SAM2 ({SAM2_CHECKPOINT}) on {device}...')
    sam2_model = build_sam2(
        config_name, ckpt_path, device=device,
        apply_postprocessing=False,
    )
    mask_generator = SAM2AutomaticMaskGenerator(
        sam2_model,
        points_per_side=32,
        pred_iou_thresh=0.86,
        stability_score_thresh=0.92,
        min_mask_region_area=200,
    )
    print('SAM2 ready.')

    def _soft_alpha(mask_f: np.ndarray, blur_r: float = 6.0) -> np.ndarray:
        pil = Image.fromarray((mask_f * 255).astype(np.uint8), 'L')
        b   = np.array(pil.filter(ImageFilter.GaussianBlur(blur_r))).astype(np.float32) / 255.0
        return 1.0 / (1.0 + np.exp(-12.0 * (b - 0.5)))

    def segment_image(image_path: str) -> bool:
        stem      = Path(image_path).stem
        bg_path   = f'{LAYERS_DIR}/{stem}_bg_rgb.png'
        mid_path  = f'{LAYERS_DIR}/{stem}_mid_mask.png'
        subj_path = f'{LAYERS_DIR}/{stem}_subj_mask.png'

        if Path(bg_path).exists() and Path(mid_path).exists() and Path(subj_path).exists():
            return True  # already cached

        img_pil  = Image.open(image_path).convert('RGB')
        img_rgb  = np.array(img_pil)
        h, w     = img_rgb.shape[:2]
        total_px = h * w

        masks = mask_generator.generate(img_rgb)
        masks.sort(key=lambda m: m['area'], reverse=True)

        bg_acc   = np.zeros((h, w), dtype=np.float32)
        mid_acc  = np.zeros((h, w), dtype=np.float32)
        subj_acc = np.zeros((h, w), dtype=np.float32)
        assigned = np.zeros((h, w), dtype=bool)

        for m in masks:
            seg = m['segmentation'].astype(bool)
            af  = m['area'] / total_px
            if   af > 0.25:  bg_acc[seg]   = 1.0
            elif af >= 0.06: mid_acc[seg]  = 1.0
            else:            subj_acc[seg] = 1.0
            assigned |= seg
        bg_acc[~assigned] = 1.0

        img_pil.save(bg_path)
        Image.fromarray((_soft_alpha(mid_acc)  * 255).astype(np.uint8), 'L').save(mid_path)
        Image.fromarray((_soft_alpha(subj_acc) * 255).astype(np.uint8), 'L').save(subj_path)
        return True

    ANCHORS_DIR = f'{DRIVE_OUTPUTS}/anchors'
    image_paths = sorted(
        glob.glob(f'{SCENE_IMAGES_DIR}/**/*.png', recursive=True) +
        glob.glob(f'{REVEAL_IMAGES_DIR}/**/*.png', recursive=True) +
        glob.glob(f'{ANCHORS_DIR}/**/*.png', recursive=True)
    )
    print(f'\nImages to segment: {len(image_paths)}')

    done = skipped = failed = 0
    for img in image_paths:
        stem = Path(img).stem
        if (Path(f'{LAYERS_DIR}/{stem}_bg_rgb.png').exists() and
                Path(f'{LAYERS_DIR}/{stem}_mid_mask.png').exists()):
            skipped += 1
            continue
        try:
            segment_image(img)
            done += 1
            print(f'  ✓ {stem}')
        except Exception as e:
            failed += 1
            print(f'  ✗ {stem}: {e}')

    print(f'\nSAM2 done: {done} new, {skipped} cached, {failed} failed')

---
## Section C — SVD-XT Motion Clips
Animates each scene image into a 4-second parallax clip.  
⚠️ **~4 compute units per run for ~17 images. Set `RUN_MOTION=True` only when images are final.**

In [ ]:
# @title C. Run motion clip generation
if not RUN_MOTION:
    print('RUN_MOTION=False — skipping.')
    print('Set RUN_MOTION=True in cell 2 when images are final (~4 compute units).')
else:
    import torch, subprocess
    from pathlib import Path
    from PIL import Image
    import glob

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    if MOTION_METHOD == 'svd':
        from diffusers import StableVideoDiffusionPipeline
        from diffusers.utils import export_to_video

        print('Loading SVD-XT pipeline...')
        pipe = StableVideoDiffusionPipeline.from_pretrained(
            'stabilityai/stable-video-diffusion-img2vid-xt',
            torch_dtype=torch.float16, variant='fp16',
        ).to(device)
        pipe.enable_model_cpu_offload()
        print('SVD-XT ready.')

        def animate_image(image_path: str, out_path: str) -> bool:
            try:
                img = Image.open(image_path).convert('RGB').resize((1024, 576))
                frames = pipe(
                    img,
                    decode_chunk_size=8,
                    generator=torch.manual_seed(42),
                    motion_bucket_id=110,
                    noise_aug_strength=0.02,
                    num_frames=25,
                ).frames[0]
                tmp = out_path.replace('.mp4', '_raw.mp4')
                export_to_video(frames, tmp, fps=6)
                subprocess.run([
                    'ffmpeg', '-y', '-i', tmp,
                    '-vf', f'scale={OUTPUT_W}:{OUTPUT_H}:flags=lanczos,setsar=1',
                    '-t', str(CLIP_DURATION), '-r', str(OUTPUT_FPS),
                    '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
                    '-preset', 'veryfast', '-b:v', '6M', out_path,
                ], capture_output=True)
                Path(tmp).unlink(missing_ok=True)
                return Path(out_path).exists()
            except Exception as e:
                print(f'  SVD error: {e}')
                return False

    elif MOTION_METHOD == 'animatediff':
        from diffusers import AnimateDiffPipeline, DDIMScheduler, MotionAdapter
        from diffusers.utils import export_to_gif

        adapter = MotionAdapter.from_pretrained(
            'guoyww/animatediff-motion-adapter-v1-5-2', torch_dtype=torch.float16)
        pipe = AnimateDiffPipeline.from_pretrained(
            'SG161222/Realistic_Vision_V5.1_noVAE',
            motion_adapter=adapter, torch_dtype=torch.float16)
        pipe.scheduler = DDIMScheduler.from_pretrained(
            'SG161222/Realistic_Vision_V5.1_noVAE', subfolder='scheduler',
            beta_schedule='linear', clip_sample=False,
            timestep_spacing='linspace', steps_offset=1)
        pipe.load_lora_weights('guoyww/animatediff-motion-lora-pan-left', adapter_name='pan_left')
        pipe.set_adapters(['pan_left'], [0.8])
        pipe.enable_vae_slicing()
        pipe = pipe.to(device)

        def animate_image(image_path: str, out_path: str) -> bool:
            try:
                stem   = Path(image_path).stem.replace('_', ' ')
                prompt = f'cinematic scene, {stem}, dark atmosphere, slow pan, film grain, 4K'
                output = pipe(prompt=prompt,
                              negative_prompt='blurry, low quality, watermark, text',
                              num_frames=16, guidance_scale=7.5,
                              num_inference_steps=25,
                              generator=torch.manual_seed(42), width=512, height=512)
                gif = out_path.replace('.mp4', '.gif')
                export_to_gif(output.frames[0], gif)
                subprocess.run([
                    'ffmpeg', '-y', '-stream_loop', '-1', '-t', str(CLIP_DURATION),
                    '-i', gif,
                    '-vf', f'scale={OUTPUT_W}:{OUTPUT_H}:flags=lanczos,setsar=1',
                    '-r', str(OUTPUT_FPS), '-c:v', 'libx264', '-pix_fmt', 'yuv420p',
                    '-preset', 'veryfast', '-b:v', '6M', out_path,
                ], capture_output=True)
                Path(gif).unlink(missing_ok=True)
                return Path(out_path).exists()
            except Exception as e:
                print(f'  AnimateDiff error: {e}')
                return False

    image_paths = sorted(
        glob.glob(f'{SCENE_IMAGES_DIR}/**/*.png', recursive=True) +
        glob.glob(f'{REVEAL_IMAGES_DIR}/**/*.png', recursive=True)
    )
    print(f'Images to animate: {len(image_paths)}')

    done = skipped = failed = 0
    for img_path in image_paths:
        stem     = Path(img_path).stem
        out_path = f'{MOTION_CLIPS_DIR}/{stem}_motion.mp4'
        if Path(out_path).exists():
            skipped += 1
            continue
        print(f'  Animating {stem}...')
        ok = animate_image(img_path, out_path)
        if ok:
            done += 1
        else:
            failed += 1
        print(f'    {"✓" if ok else "✗"} {out_path}')

    print(f'\nMotion done: {done} new, {skipped} cached, {failed} failed')

---
## After running this notebook

### 1. Sync back to local machine
Download (or rclone) the relevant sub-folders from `DRIVE_OUTPUTS`:

| What ran | Folder to download |
|----------|-------------------|
| VOICE    | `outputs/audio/episode_N_voiceover.wav` |
| SAM2     | `outputs/layers/` (entire folder) |
| MOTION   | `outputs/motion_clips/` (entire folder) |

### 2. Update your local `.env`
```
VOICE_SOURCE=colab   # skip DashScope TTS; use the cloned WAV
SAM2_SOURCE=colab    # skip local SAM2; use layer PNGs from Colab
```
Leave unset (or `local`) to use local computation instead.

### 3. Re-run local pipeline
The pipeline resumes from Agent 4 (voiceover registers the existing WAV),  
Whisper alignment runs locally on the cloned audio,  
then the rest continues as normal.

### File naming (must match exactly)
| Type | Expected path |
|------|---------------|
| Cloned voice | `outputs/audio/episode_N_voiceover.wav` |
| Layer bg     | `outputs/layers/{stem}_bg_rgb.png` |
| Layer mid    | `outputs/layers/{stem}_mid_mask.png` |
| Layer subj   | `outputs/layers/{stem}_subj_mask.png` |
| Motion clip  | `outputs/motion_clips/{stem}_motion.mp4` |